# Ablation Variant B: SmallBERT + Jacobian Norm Penalty

**Core Question**: What if we add explicit smoothness constraints to the
Transformer? Would that close the geometric stability gap with SSMs?

**The Goal**: Prove that forcing local smoothness via Jacobian regularization
comes at the cost of **catastrophic predictive collapse** -- the exact
manifestation of the Geometric Alignment Tax.

## Design

- Keep the **discrete categorical head** (256 bins + CrossEntropy)
- Add a **Jacobian norm penalty**: minimize `||dh/dx||_F` where `h` is the
  hidden state and `x` is the input embedding
- This explicitly penalizes large changes in hidden representations for
    small input changes -- an explicit geometric smoothness constraint
- Train with: `L_total = L_CE + lambda * L_jacobian`

**Expected Result**: As lambda increases, a divergence between geometric stability and task performance:
- Composite stability may improve slightly
- Validation CE **explodes** -- predictive collapse

This proves the Tax is a **zero-sum tradeoff**: you cannot have
both smooth geometry AND accurate attention-based predictions.

## Prerequisites

- Upload `evaluation_harness.py` to `/content/`
- GPU runtime required
- Run after `Architectural_Control_Experiment.ipynb` (for baseline comparison)

---

In [ ]:
# Install Dependencies

print('Installing core dependencies...')
!pip install -q torch shesha-geometry matplotlib seaborn pandas scipy

print('\nBuilding mamba-ssm from source...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: OK (native CUDA)')
except ImportError:
    print('mamba-ssm: FAILED -- will use pure-PyTorch SSM fallback')

import torch
print(f'\ntorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

In [ ]:
# Configuration

import os, sys, gc, time
import numpy as np
import pandas as pd
sys.path.insert(0, '.')

OUTPUT_BASE = './results/ablation_variant_b_jacobian/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

SEQ_LEN    = 512
N_BINS     = 256
N_TRAIN    = 50_000
N_EVAL     = 2_000
SEED       = 320
PAD_TOKEN  = N_BINS
MASK_TOKEN = N_BINS + 1
VOCAB_SIZE = N_BINS + 2

D_MODEL    = 256
N_LAYERS   = 4
N_HEADS    = 4
FFN_DIM    = 1024
D_STATE    = 16
D_CONV     = 4
EXPAND     = 2

LR         = 3e-4
WEIGHT_DECAY = 0.01
EPOCHS     = 20
BATCH_SIZE = 64
NOISE_RATES = [0.01, 0.02, 0.05, 0.10]
DATASETS = ['waveform', 'oscillator', 'lorenz']

# Jacobian penalty sweep: test multiple lambda values
JACOBIAN_LAMBDAS = [0.001, 0.01, 0.1, 1.0]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Ablation: Variant B -- Jacobian Norm Penalty')
print(f'Lambda sweep: {JACOBIAN_LAMBDAS}')
print(f'Output: {OUTPUT_BASE}')

In [ ]:
# Dataset Generators (FIX v2: global discretization)

from scipy.integrate import solve_ivp

def discretize(values, n_bins=N_BINS, global_min=None, global_max=None):
    '''Uses dataset-global min/max when provided.'''
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12:
        return np.full_like(values, n_bins // 2, dtype=np.int64)
    normed = np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0)
    return np.clip((normed * (n_bins - 1)).astype(np.int64), 0, n_bins - 1)

def generate_waveforms(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 1, seq_len, endpoint=False)
    raw = []
    for _ in range(n_sequences):
        n_sines = rng.integers(3, 6)
        signal = np.zeros(seq_len)
        for _ in range(n_sines):
            signal += rng.uniform(0.1, 1.0) * np.sin(
                2*np.pi * rng.uniform(0.5, 50.0) * t + rng.uniform(0, 2*np.pi))
        raw.append(signal)
    if global_range is None:
        all_v = np.concatenate(raw)
        global_range = (float(all_v.min()), float(all_v.max()))
    gmin, gmax = global_range
    seqs = [discretize(s, global_min=gmin, global_max=gmax) for s in raw]
    return np.array(seqs, dtype=np.int64), global_range

def generate_oscillators(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 4, seq_len, endpoint=False)
    raw = []
    for _ in range(n_sequences):
        signal = rng.uniform(0.5,2)*np.exp(-rng.uniform(0.2,2)*t)*np.cos(
            rng.uniform(2,20)*t + rng.uniform(0,2*np.pi))
        raw.append(signal)
    if global_range is None:
        all_v = np.concatenate(raw)
        global_range = (float(all_v.min()), float(all_v.max()))
    gmin, gmax = global_range
    seqs = [discretize(s, global_min=gmin, global_max=gmax) for s in raw]
    return np.array(seqs, dtype=np.int64), global_range

def _lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = state
    return [sigma*(y-x), x*(rho-z)-y, x*y-beta*z]

def generate_lorenz(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed)
    t_span = (0, 25)
    t_eval = np.linspace(*t_span, seq_len)
    raw = []
    for _ in range(n_sequences):
        x0,y0,z0 = rng.uniform(-15,15), rng.uniform(-15,15), rng.uniform(10,40)
        sol = solve_ivp(_lorenz_rhs, t_span, [x0,y0,z0], t_eval=t_eval, method='RK45', max_step=0.05)
        if sol.success and len(sol.y[0]) == seq_len:
            raw.append(sol.y[0])
        else:
            sol2 = solve_ivp(_lorenz_rhs, t_span, [x0+rng.uniform(-1,1),y0,z0],
                             t_eval=t_eval, method='RK45', max_step=0.01)
            sig = sol2.y[0][:seq_len]
            if len(sig) < seq_len:
                sig = np.pad(sig, (0, seq_len - len(sig)), mode='edge')
            raw.append(sig)
    if global_range is None:
        all_v = np.concatenate(raw)
        global_range = (float(all_v.min()), float(all_v.max()))
    gmin, gmax = global_range
    seqs = [discretize(s, global_min=gmin, global_max=gmax) for s in raw]
    return np.array(seqs, dtype=np.int64), global_range

GENERATORS = {'waveform': generate_waveforms, 'oscillator': generate_oscillators, 'lorenz': generate_lorenz}
GLOBAL_RANGES = {}
print('Dataset generators ready (global discretization)')


In [ ]:
# Perturbation Suite (identical to baseline)

from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: np.ndarray
    params: dict = field(default_factory=dict)
    description: str = ''

def noise_perturb(sequences, rate, rng, n_bins=N_BINS):
    out = sequences.copy()
    mask = rng.random(out.shape) < rate
    noise = rng.normal(0, n_bins*0.10, size=out.shape).astype(np.int64)
    out[mask] = np.clip(out[mask] + noise[mask], 0, n_bins - 1)
    return out

class ContinuousPerturbationSuite:
    def __init__(self, seed=SEED, noise_rates=None):
        self.rng = np.random.default_rng(seed)
        self.noise_rates = noise_rates or NOISE_RATES
    def run_all(self, sequences):
        results = {}
        for rate in self.noise_rates:
            name = f'noise_{int(rate*100)}pct'
            results[name] = PerturbedSet(name=name, category='noise',
                sequences=noise_perturb(sequences, rate, self.rng),
                params={'noise_rate': rate})
        results['time_reversal'] = PerturbedSet(name='time_reversal', category='reversal',
            sequences=sequences[:, ::-1].copy())
        return results

print('ContinuousPerturbationSuite ready')

In [ ]:
# Model Definition -- SmallBERT with Jacobian-Aware Forward
#
# IMPORTANT: When compute_jacobian=True, we force the MATH SDPA backend
# because Flash/Efficient attention doesn't support double-backward
# (needed for autograd.grad with create_graph=True).

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class SmallBERT_Jacobian(nn.Module):
    """SmallBERT with optional Jacobian norm computation.
    
    The Jacobian penalty encourages:
        ||dh/dx||_F -> 0
    where h is the hidden state and x is the input embedding.
    This is an explicit local smoothness constraint."""

    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN,
                 dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size)  # standard discrete head
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, x, return_hidden=False, compute_jacobian=False):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        
        # Get embeddings -- need grad for Jacobian
        emb = self.tok_emb(x) + self.pos_emb(pos)
        if compute_jacobian:
            emb = emb.detach().requires_grad_(True)
        h = self.drop(emb)
        
        causal_mask = self._causal_mask(L, x.device)
        
        if compute_jacobian:
            # Flash/Efficient SDPA doesn't support double-backward derivatives.
            # Force the MATH backend which implements full autograd.
            with torch.nn.attention.sdpa_kernel(
                    [torch.nn.attention.SDPBackend.MATH]):
                h = self.encoder(h, mask=causal_mask)
        else:
            h = self.encoder(h, mask=causal_mask)
        
        h = self.norm(h)
        logits = self.head(h)
        
        if compute_jacobian:
            # Approximate Jacobian norm via random projection (Hutchinson's trick)
            # ||J||_F^2 ~ E[||J^T v||^2] where v ~ N(0, I)
            v = torch.randn_like(h)
            jvp = torch.autograd.grad(
                outputs=h, inputs=emb,
                grad_outputs=v,
                create_graph=True, retain_graph=True,
            )[0]
            jacobian_norm = (jvp ** 2).mean()
            
            if return_hidden:
                return logits, h, jacobian_norm
            return logits, jacobian_norm
        
        if return_hidden:
            return logits, h
        return logits


# Print param count
m = SmallBERT_Jacobian()
n_params = sum(p.numel() for p in m.parameters())
print(f'SmallBERT_Jacobian: {n_params/1e6:.2f}M params')
del m
print('Model ready (Jacobian-aware forward with MATH SDPA backend)')

# =============================================================================
# SmallStripedHyena (Evo 2 architecture: Hyena conv + Attention stripes)
# =============================================================================

class ImplicitFilterMLP(nn.Module):
    """Learns long convolution filter implicitly via MLP (Hyena's key innovation)."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        n_pos_features = 16
        self.pos_emb = nn.Linear(n_pos_features, n_hidden)
        self.mlp = nn.Sequential(nn.GELU(), nn.Linear(n_hidden, n_hidden), nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features', self._make_pos_features(seq_len, n_pos_features))

    def _make_pos_features(self, seq_len, n_features):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n_features).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        if seq_len != self.seq_len:
            pf = self._make_pos_features(seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        else:
            pf = self.pos_features
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    """Data-controlled long convolution with gating (replaces attention)."""
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=short_conv_kernel,
                      padding=short_conv_kernel // 2, groups=d_model) for _ in range(order + 1)])
        self.filters = nn.ModuleList([ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]
        fft_len = 2 * L
        return torch.fft.irfft(
            torch.fft.rfft(signal, n=fft_len, dim=-1) * torch.fft.rfft(kernel, n=fft_len, dim=-1).unsqueeze(0),
            n=fft_len, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv_outputs = [self.short_convs[i](branches[:, :, i, :].transpose(1, 2)) for i in range(self.order + 1)]
        v = conv_outputs[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv_outputs[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Standard multi-head attention with RoPE (for StripedHyena minority layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs', torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _apply_rotary(self, x, freqs):
        L = x.shape[2]
        freqs = freqs[:L]
        cos_f = torch.cos(freqs).unsqueeze(0).unsqueeze(0)
        sin_f = torch.sin(freqs).unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f], dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q, k = self._apply_rotary(q, self.freqs), self._apply_rotary(k, self.freqs)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn = F.softmax(attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf')), dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    """Single StripedHyena block: Hyena or Attention + SwiGLU MLP."""
    def __init__(self, d_model, seq_len, n_heads=4, order=2, is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mixer = SHMultiHeadAttention(d_model, n_heads) if is_attention else HyenaOperator(d_model, seq_len, order=order)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp_gate = nn.Linear(d_model, mlp_hidden)
        self.mlp_value = nn.Linear(d_model, mlp_hidden)
        self.mlp_out = nn.Linear(mlp_hidden, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_out(F.silu(self.mlp_gate(h)) * self.mlp_value(h))
        return x


class SmallStripedHyena(nn.Module):
    """
    Minimal StripedHyena (Evo 2 arch) for bottleneck isolation.
    ~3.4M params: d_model=128, 8 layers (6 Hyena + 2 Attention), seq_len=512.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model, self.vocab_size = d_model, vocab_size
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        # No weight tying (vocab_size=258 != d_model=256)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


class SmallStripedHyena_Jacobian(nn.Module):
    """SmallStripedHyena with optional Jacobian norm computation.

    Mirrors SmallBERT_Jacobian: when compute_jacobian=True, computes
    ||dh/dx||_F via Hutchinson's trace estimator on the embedding->hidden map.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model, self.vocab_size = d_model, vocab_size
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order,
                              is_attention=(i in self.attention_layers),
                              mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        # No weight tying (vocab_size=258 != d_model=256)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena_Jacobian: {n_p:.2f}M params '
              f'({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False, compute_jacobian=False):
        emb = self.tok_emb(x)
        if compute_jacobian:
            emb = emb.detach().requires_grad_(True)
        h = self.drop(emb)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)

        if compute_jacobian:
            v = torch.randn_like(h)
            jvp = torch.autograd.grad(
                outputs=h, inputs=emb,
                grad_outputs=v,
                create_graph=True, retain_graph=True,
            )[0]
            jacobian_norm = (jvp ** 2).mean()
            if return_hidden:
                return logits, h, jacobian_norm
            return logits, jacobian_norm

        if return_hidden:
            return logits, h
        return logits


# Print Hyena param counts
m_sh = SmallStripedHyena_Jacobian()
del m_sh
print('SmallStripedHyena_Jacobian ready (Jacobian-aware forward)')


In [ ]:
# Training Loop -- CLM + Jacobian Penalty
#
# L_total = L_CE + lambda * L_jacobian
#
# The Jacobian penalty is computed every N batches (expensive)
# to keep training tractable.
#
# Returns: (model, arch_name, train_losses, val_losses, best_val_ce)
#   best_val_ce is the minimum validation CE achieved during training.
#   This is CRITICAL for proving predictive collapse.

from torch.utils.data import DataLoader, TensorDataset


def create_causal_batch(x):
    return x[:, :-1], x[:, 1:]


def train_model_jacobian(model, train_data, val_data, dataset_name,
                         jacobian_lambda, epochs=EPOCHS, batch_size=BATCH_SIZE,
                         lr=LR, weight_decay=WEIGHT_DECAY, seed=SEED,
                         jacobian_every_n=4, arch_prefix='SmallBERT_Jacobian'):
    """Train SmallBERT with Jacobian norm penalty.
    
    Returns:
        model, arch_name, train_losses, val_losses, best_val_ce
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = model.to(DEVICE)
    train_ds = TensorDataset(torch.from_numpy(train_data).long())
    val_ds   = TensorDataset(torch.from_numpy(val_data).long())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader))
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_state = None
    train_losses, val_losses = [], []

    lam_str = f'{jacobian_lambda:.0e}' if jacobian_lambda < 1 else f'{jacobian_lambda:.1f}'
    arch_name = f'{arch_prefix}_lam{lam_str}'
    ckpt_path = f'{CKPT_DIR}/{arch_name}_{dataset_name}_seed{seed}_best.pt'

    if os.path.exists(ckpt_path):
        print(f'  Loading existing checkpoint: {ckpt_path}')
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
        # Re-evaluate validation CE so we always have the number
        model.eval()
        val_loss_sum, val_batches = 0.0, 0
        with torch.no_grad():
            for (batch_x,) in val_loader:
                batch_x = batch_x.to(DEVICE)
                inputs, targets = create_causal_batch(batch_x)
                logits = model(inputs)
                loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
                val_loss_sum += loss.item()
                val_batches += 1
        loaded_val_ce = val_loss_sum / max(val_batches, 1)
        print(f'  Checkpoint val CE: {loaded_val_ce:.4f}')
        return model, arch_name, [], [loaded_val_ce], loaded_val_ce

    print(f'  Training {arch_name} on {dataset_name} (lambda={jacobian_lambda}, {epochs} epochs)...')
    start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_ce, epoch_jac, n_batches = 0.0, 0.0, 0

        for batch_idx, (batch_x,) in enumerate(train_loader):
            batch_x = batch_x.to(DEVICE)
            inputs, targets = create_causal_batch(batch_x)

            use_jacobian = (batch_idx % jacobian_every_n == 0)

            if use_jacobian:
                logits, jac_norm = model(inputs, compute_jacobian=True)
                ce_loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
                total_loss = ce_loss + jacobian_lambda * jac_norm
                epoch_jac += jac_norm.item()
            else:
                logits = model(inputs)
                ce_loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
                total_loss = ce_loss

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            epoch_ce += ce_loss.item()
            n_batches += 1

        avg_ce = epoch_ce / n_batches
        avg_jac = epoch_jac / max(1, n_batches // jacobian_every_n)
        train_losses.append(avg_ce)

        # Validation (CE only)
        model.eval()
        val_loss, val_batches = 0.0, 0
        with torch.no_grad():
            for (batch_x,) in val_loader:
                batch_x = batch_x.to(DEVICE)
                inputs, targets = create_causal_batch(batch_x)
                logits = model(inputs)
                loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
                val_loss += loss.item()
                val_batches += 1

        avg_val = val_loss / max(val_batches, 1)
        val_losses.append(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0 or epoch == 0:
            elapsed = time.time() - start
            print(f'    Epoch {epoch+1:3d}/{epochs}  '
                  f'ce={avg_ce:.4f}  jac={avg_jac:.4f}  '
                  f'val_ce={avg_val:.4f}  [{elapsed:.0f}s]')

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt_path)
    elapsed = time.time() - start
    print(f'  Done in {elapsed/60:.1f} min  (best val CE: {best_val_loss:.4f})')
    return model, arch_name, train_losses, val_losses, best_val_loss


print('Training loop ready (CLM + Jacobian penalty, returns best_val_ce)')

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness, ModelReport

harness = StabilityHarness(
    window_size=0, metric='cosine', n_splits=30,
    seed=320, max_samples=2500, n_bootstrap=5,
)
print('Shesha StabilityHarness configured')

In [ ]:
# Run Lambda Sweep -- BERT + Hyena
#
# For each lambda value, train both SmallBERT_Jacobian and
# SmallStripedHyena_Jacobian on all 3 datasets and evaluate
# geometric stability AND predictive quality (val CE).
#
# Additionally, run SmallStripedHyena_Jacobian at lambda=0
# as a no-penalty baseline reference.

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

@torch.no_grad()
def extract_embeddings(model, sequences, batch_size=128):
    model.eval()
    all_embs = []
    for i in range(0, len(sequences), batch_size):
        batch = torch.from_numpy(sequences[i:i+batch_size]).long().to(DEVICE)
        _, hidden = model(batch, return_hidden=True)
        emb = hidden.mean(dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.concatenate(all_embs, axis=0)


# Build sweep configs: (model_class, arch_prefix, lambda_values)
SWEEP_CONFIGS = [
    (SmallBERT_Jacobian, 'SmallBERT_Jacobian', JACOBIAN_LAMBDAS),
    (SmallStripedHyena_Jacobian, 'SmallStripedHyena_Jacobian',
     [0.0] + JACOBIAN_LAMBDAS),  # 0.0 = no-penalty baseline
]

print('=' * 70)
print('ABLATION VARIANT B: JACOBIAN NORM PENALTY -- LAMBDA SWEEP')
print('  Models: SmallBERT_Jacobian + SmallStripedHyena_Jacobian')
print('  Hyena includes lambda=0 (no-penalty baseline)')
print('Tracking BOTH stability AND predictive quality (val CE)')
print('=' * 70)

all_detailed_rows = []

for model_cls, arch_prefix, lambda_list in SWEEP_CONFIGS:
    print(f"\n{'*'*70}")
    print(f'MODEL: {arch_prefix}')
    print(f'  Lambdas: {lambda_list}')
    print('*' * 70)

    for jac_lambda in lambda_list:
        print(f"\n{'#'*70}")
        print(f'{arch_prefix} | JACOBIAN LAMBDA = {jac_lambda}')
        print('#' * 70)

        for ds_name in DATASETS:
            gen_fn = GENERATORS[ds_name]
            print(f"\n  {'='*60}")
            print(f'  DATASET: {ds_name.upper()} | {arch_prefix} | lambda = {jac_lambda}')
            print('  ' + '=' * 60)

            train_data, grange = gen_fn(N_TRAIN, seed=SEED)
            GLOBAL_RANGES[ds_name] = grange
            eval_data, _ = gen_fn(N_EVAL, seed=SEED + 1000, global_range=grange)
            pert_suite = ContinuousPerturbationSuite(seed=SEED)
            perturbed_sets = pert_suite.run_all(eval_data)

            model = model_cls()

            if jac_lambda == 0.0:
                # No-penalty baseline: train with lambda=0 (pure CE)
                model, arch_name, t_losses, v_losses, best_val_ce = train_model_jacobian(
                    model, train_data, eval_data, ds_name,
                    jacobian_lambda=0.0, seed=SEED,
                    arch_prefix=f'{arch_prefix}_baseline'
                )
            else:
                model, arch_name, t_losses, v_losses, best_val_ce = train_model_jacobian(
                    model, train_data, eval_data, ds_name,
                    jacobian_lambda=jac_lambda, seed=SEED,
                    arch_prefix=arch_prefix
                )

            condition_key = f'{arch_name}_{ds_name}'

            # Extract embeddings
            cache_clean = f'{CACHE_DIR}/{condition_key}_clean.npy'
            if os.path.exists(cache_clean):
                embeddings_clean = np.load(cache_clean)
            else:
                embeddings_clean = extract_embeddings(model, eval_data)
                np.save(cache_clean, embeddings_clean)

            perturbed_embeddings = {}
            for pert_name, pset in perturbed_sets.items():
                cache_pert = f'{CACHE_DIR}/{condition_key}_{pert_name}.npy'
                if os.path.exists(cache_pert):
                    perturbed_embeddings[pert_name] = np.load(cache_pert)
                else:
                    perturbed_embeddings[pert_name] = extract_embeddings(model, pset.sequences)
                    np.save(cache_pert, perturbed_embeddings[pert_name])

            report = harness.evaluate_all_perturbations(
                model_name=condition_key,
                embeddings_clean=embeddings_clean,
                perturbed_dict=perturbed_embeddings,
            )

            for pert_name, metrics in report.perturbation_breakdown().items():
                all_detailed_rows.append({
                    'dataset': ds_name,
                    'architecture': arch_name,
                    'model_family': arch_prefix,
                    'variant': 'VariantB_Jacobian',
                    'jacobian_lambda': jac_lambda,
                    'perturbation': pert_name,
                    'rdm_similarity': metrics.get('rdm_similarity_score', 0),
                    'pert_stability': metrics.get('perturbation_stability_score', 0),
                    'composite': metrics.get('composite_stability', 0),
                    'val_ce': best_val_ce,
                })

            summary = report.summary()
            print(f'    Composite: {summary["mean_composite_stability"]:.4f}  |  Val CE: {best_val_ce:.4f}')

            del model
            cleanup_gpu()

df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/variant_b_jacobian_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nSaved: {detailed_path}')
print('\nSummary by model family + lambda:')
print(df_detailed.groupby(['model_family', 'jacobian_lambda'])[['composite', 'val_ce']].mean().to_string())


In [ ]:
# Visualization -- Lambda Sweep Results
#
# Panel A: Stability vs Lambda (both architectures + baseline references)
# Panel B: The Zero-Sum Tradeoff (dual-axis: stability vs CE, both archs)
# Panel C: Per-dataset breakdown

import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11})

# Load baseline for comparison
baseline_path = './results/architectural_control_experiment/results/architectural_control_detailed_v2.csv'
if not os.path.exists(baseline_path):
    baseline_path = './results/architectural_control_experiment/results/architectural_control_detailed.csv'
has_baseline = os.path.exists(baseline_path)
if has_baseline:
    df_baseline = pd.read_csv(baseline_path)
    bert_baseline_comp = df_baseline[df_baseline['architecture'] == 'SmallBERT']['composite'].mean()
    mamba_baseline_comp = df_baseline[df_baseline['architecture'] == 'SmallMamba']['composite'].mean()
    print(f'Baseline BERT composite: {bert_baseline_comp:.4f}')
    print(f'Baseline Mamba composite: {mamba_baseline_comp:.4f}')
else:
    bert_baseline_comp = None
    mamba_baseline_comp = None

# Separate BERT and Hyena results (exclude Hyena lambda=0 baseline from sweep plots)
df_bert = df_detailed[df_detailed['model_family'] == 'SmallBERT_Jacobian']
df_hyena_sweep = df_detailed[
    (df_detailed['model_family'] == 'SmallStripedHyena_Jacobian') &
    (df_detailed['jacobian_lambda'] > 0)
]
df_hyena_baseline = df_detailed[
    df_detailed['model_family'] == 'SmallStripedHyena_Jacobian_baseline'
]
hyena_baseline_comp = df_hyena_baseline['composite'].mean() if len(df_hyena_baseline) > 0 else None

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# ---- Panel A: Composite Stability vs Lambda ----
ax = axes[0]

# BERT sweep
agg_bert = df_bert.groupby('jacobian_lambda')['composite'].agg(['mean', 'std']).reset_index()
ax.errorbar(agg_bert['jacobian_lambda'], agg_bert['mean'], yerr=agg_bert['std'],
            marker='o', linewidth=2.5, markersize=10, capsize=5,
            color='#9333EA', label='SmallBERT + Jacobian')

# Hyena sweep
if len(df_hyena_sweep) > 0:
    agg_hyena = df_hyena_sweep.groupby('jacobian_lambda')['composite'].agg(['mean', 'std']).reset_index()
    ax.errorbar(agg_hyena['jacobian_lambda'], agg_hyena['mean'], yerr=agg_hyena['std'],
                marker='D', linewidth=2.5, markersize=9, capsize=5,
                color='#7C3AED', label='SmallStripedHyena + Jacobian')

# Baselines
if bert_baseline_comp is not None:
    ax.axhline(y=bert_baseline_comp, color='#2563EB', linestyle='--',
               linewidth=2, alpha=0.7, label=f'Baseline BERT ({bert_baseline_comp:.3f})')
if mamba_baseline_comp is not None:
    ax.axhline(y=mamba_baseline_comp, color='#16A34A', linestyle='--',
               linewidth=2, alpha=0.7, label=f'Baseline Mamba ({mamba_baseline_comp:.3f})')
if hyena_baseline_comp is not None:
    ax.axhline(y=hyena_baseline_comp, color='#F59E0B', linestyle=':',
               linewidth=2, alpha=0.7, label=f'Hyena no-penalty ({hyena_baseline_comp:.3f})')

ax.set_xscale('log')
ax.set_xlabel('Jacobian Penalty lambda', fontsize=12)
ax.set_ylabel('Mean Composite Stability', fontsize=12)
ax.set_title('A. Stability vs Smoothness Penalty', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
ax.set_ylim(0, 1.05)

# ---- Panel B: Stability vs. Task Performance Divergence (dual-axis, both archs) ----
ax2 = axes[1]
ax2_twin = ax2.twinx()

for df_sub, label_prefix, color_stab, color_ce, marker in [
    (df_bert, 'BERT', '#2563EB', '#DC2626', 'o'),
    (df_hyena_sweep, 'Hyena', '#7C3AED', '#F97316', 'D'),
]:
    if len(df_sub) == 0:
        continue
    tradeoff = df_sub.groupby('jacobian_lambda').agg(
        mean_composite=('composite', 'mean'),
        mean_val_ce=('val_ce', 'mean'),
    ).reset_index()
    ax2.plot(tradeoff['jacobian_lambda'], tradeoff['mean_composite'],
             f'{marker}-', linewidth=2.5, markersize=9, color=color_stab,
             label=f'{label_prefix} Stability')
    ax2_twin.plot(tradeoff['jacobian_lambda'], tradeoff['mean_val_ce'],
                  f'{marker}--', linewidth=2.5, markersize=9, color=color_ce,
                  label=f'{label_prefix} Val CE')

ax2.set_ylabel('Composite Stability\n(Higher = Smoother Manifold)', color='#2563EB', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#2563EB')
ax2.set_ylim(0, 1.05)
ax2_twin.set_ylabel('Validation CE\n(Lower = Better Prediction)', color='#DC2626', fontsize=11)
ax2_twin.tick_params(axis='y', labelcolor='#DC2626')
ax2.set_xscale('log')
ax2.set_xlabel('Jacobian Penalty lambda', fontsize=12)
ax2.set_title('B. The Zero-Sum Tradeoff\n(Predictive Collapse)', fontweight='bold')

# Combined legend
lines_a, labels_a = ax2.get_legend_handles_labels()
lines_b, labels_b = ax2_twin.get_legend_handles_labels()
ax2.legend(lines_a + lines_b, labels_a + labels_b, loc='center left', fontsize=8)
ax2.grid(alpha=0.2)

# ---- Panel C: Per-dataset breakdown (BERT only for clarity) ----
ax = axes[2]
ds_colors = {'waveform': '#2563EB', 'oscillator': '#16A34A', 'lorenz': '#DC2626'}
for ds_name in DATASETS:
    sub = df_bert[df_bert['dataset'] == ds_name]
    ds_agg = sub.groupby('jacobian_lambda')['composite'].mean().reset_index()
    ax.plot(ds_agg['jacobian_lambda'], ds_agg['composite'], 'o-',
            linewidth=2, markersize=8, color=ds_colors[ds_name], label=f'BERT {ds_name}')
    # Hyena per-dataset
    sub_h = df_hyena_sweep[df_hyena_sweep['dataset'] == ds_name]
    if len(sub_h) > 0:
        ds_agg_h = sub_h.groupby('jacobian_lambda')['composite'].mean().reset_index()
        ax.plot(ds_agg_h['jacobian_lambda'], ds_agg_h['composite'], 'D--',
                linewidth=1.5, markersize=6, color=ds_colors[ds_name], alpha=0.6,
                label=f'Hyena {ds_name}')

ax.set_xscale('log')
ax.set_xlabel('Jacobian Penalty lambda', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('C. Per-Dataset Lambda Sweep', fontweight='bold')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.2)
ax.set_ylim(0, 1.05)

fig.suptitle('Ablation Variant B: Jacobian Norm Penalty\n'
             'Lambda Sweep Results
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig_path = f'{RESULTS_DIR}/variant_b_lambda_sweep.{ext}'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {RESULTS_DIR}/variant_b_lambda_sweep.png/.pdf')


In [ ]:
# Summary

print('=' * 70)
print('VARIANT B RESULTS SUMMARY')
print('=' * 70)

for family_label, family_key in [('SmallBERT', 'SmallBERT_Jacobian'),
                                  ('SmallStripedHyena', 'SmallStripedHyena_Jacobian')]:
    df_fam = df_detailed[df_detailed['model_family'] == family_key]
    if len(df_fam) == 0:
        continue
    print(f'\n--- {family_label} ---')
    print(f'  {"lambda":>10s}  {"Composite":>12s}  {"Val CE":>10s}  {"Verdict":>20s}')
    print('  ' + '-' * 60)

    fam_lambdas = sorted(df_fam['jacobian_lambda'].unique())
    baseline_ce_fam = df_fam[df_fam['jacobian_lambda'] == fam_lambdas[0]]['val_ce'].mean()

    for lam in fam_lambdas:
        sub = df_fam[df_fam['jacobian_lambda'] == lam]
        comp_m = sub['composite'].mean()
        comp_s = sub['composite'].std()
        ce_m = sub['val_ce'].mean()
        if lam == fam_lambdas[0]:
            verdict = '(baseline-like)'
        elif ce_m > baseline_ce_fam * 1.5:
            verdict = 'PREDICTIVE COLLAPSE'
        else:
            verdict = 'degrading...'
        print(f'  {lam:>10.3f}  {comp_m:>8.4f}+/-{comp_s:.4f}  {ce_m:>10.4f}  {verdict:>20s}')

# Hyena no-penalty baseline
df_hyena_bl = df_detailed[df_detailed['model_family'] == 'SmallStripedHyena_Jacobian_baseline']
if len(df_hyena_bl) > 0:
    print(f'\n--- SmallStripedHyena no-penalty baseline ---')
    print(f'  Composite: {df_hyena_bl["composite"].mean():.4f} +/- {df_hyena_bl["composite"].std():.4f}')
    print(f'  Val CE:    {df_hyena_bl["val_ce"].mean():.4f}')

# Cross-architecture comparison
print('\n' + '=' * 70)
print('CROSS-ARCHITECTURE COMPARISON')
print('=' * 70)

df_bert_sweep = df_detailed[df_detailed['model_family'] == 'SmallBERT_Jacobian']
df_hyena_sweep = df_detailed[
    (df_detailed['model_family'] == 'SmallStripedHyena_Jacobian') &
    (df_detailed['jacobian_lambda'] > 0)
]

if len(df_bert_sweep) > 0:
    best_bert_lam = df_bert_sweep.groupby('jacobian_lambda')['composite'].mean().idxmax()
    best_bert_comp = df_bert_sweep.groupby('jacobian_lambda')['composite'].mean().max()
    best_bert_ce = df_bert_sweep[df_bert_sweep['jacobian_lambda'] == best_bert_lam]['val_ce'].mean()
    print(f'  BERT best lambda={best_bert_lam}: composite={best_bert_comp:.4f}, CE={best_bert_ce:.4f}')

if len(df_hyena_sweep) > 0:
    best_hyena_lam = df_hyena_sweep.groupby('jacobian_lambda')['composite'].mean().idxmax()
    best_hyena_comp = df_hyena_sweep.groupby('jacobian_lambda')['composite'].mean().max()
    best_hyena_ce = df_hyena_sweep[df_hyena_sweep['jacobian_lambda'] == best_hyena_lam]['val_ce'].mean()
    print(f'  Hyena best lambda={best_hyena_lam}: composite={best_hyena_comp:.4f}, CE={best_hyena_ce:.4f}')

if len(df_hyena_bl) > 0:
    hyena_bl_comp = df_hyena_bl['composite'].mean()
    hyena_bl_ce = df_hyena_bl['val_ce'].mean()
    print(f'  Hyena no-penalty baseline: composite={hyena_bl_comp:.4f}, CE={hyena_bl_ce:.4f}')